# Лабораторна робота №6 (Варіант 2)
## Багатокрокове планування та автономний виклик інструментів

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Автономна оркестрація ланцюжка інструментів (Autonomous Multi-Step Tool Chaining)  

### Мета роботи
Дослідити здатність агента LlamaIndex здійснювати багатокрокове автономне планування розв'язання задач діагностики інцидентів у промислових моделях. Реалізувати перехоплення потоку подій (`stream_events`) для реєстрації та збереження детальних структурних трейсів виконання (`ToolCall`, `ToolCallResult`).

In [1]:
import os
import json
import asyncio
from pathlib import Path
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.agent.workflow import AgentWorkflow, ToolCall, ToolCallResult
from llama_index.core.memory import ChatMemoryBuffer

def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'data').exists() and (current / 'outputs').exists():
        return current
    if (current / 'lab6_llamaindex_agents').exists():
        return current / 'lab6_llamaindex_agents'
    for p in [current] + list(current.parents):
        cand = p / 'lab6_llamaindex_agents'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
DATA_DIR = LAB_DIR / 'data'
OUTPUTS_DIR = LAB_DIR / 'outputs'
TRACES_DIR = OUTPUTS_DIR / 'traces'
TRACES_DIR.mkdir(parents=True, exist_ok=True)

# Підключення перевірених інструментів
def query_model_registry(model_id: str) -> str:
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for m in data['models']:
        if m['model_id'].lower() == model_id.strip().lower():
            return json.dumps(m, indent=2)
    return f"Model '{model_id}' not found."

def list_degraded_models() -> str:
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return json.dumps([m for m in data['models'] if m['status'] != 'healthy'], indent=2)

def consult_incident_runbook(query: str) -> str:
    runbook_path = DATA_DIR / 'mlops_runbook.txt'
    with open(runbook_path, 'r', encoding='utf-8') as f:
        content = f.read()
    sections = content.split('## ')
    matched = []
    q_words = set(query.lower().replace('_', ' ').split())
    for sec in sections[1:]:
        if any(w in sec.lower() for w in q_words):
            matched.append('## ' + sec.strip())
    return '\n\n'.join(matched[:2]) if matched else content[:1200]

def calculate_slo_margin(metric_value: float, slo_target: float, metric_type: str = 'latency') -> str:
    diff = metric_value - slo_target
    pct = (diff / slo_target) * 100.0 if metric_type.lower() == 'latency' else diff
    status = 'VIOLATION' if diff > 0 else 'COMPLIANT'
    return json.dumps({
        'metric_type': metric_type,
        'value': metric_value,
        'slo_target': slo_target,
        'diff': round(diff, 2),
        'margin_pct': round(pct, 2),
        'status': status
    })

llm = OpenAILike(
    model='qwen3:4b-instruct',
    api_base='http://127.0.0.1:11434/v1',
    api_key='ollama',
    is_chat_model=True,
    is_function_calling_model=True,
    context_window=4096,
    temperature=0.1
)

agent = AgentWorkflow.from_tools_or_functions(
    [query_model_registry, list_degraded_models, consult_incident_runbook, calculate_slo_margin],
    llm=llm,
    system_prompt='You are an expert MLOps Incident Copilot. Use tools sequentially to diagnose issues and find runbook remedies.',
    verbose=False
)
print('Агент готовий до багатокрокових досліджень.')

Агент готовий до багатокрокових досліджень.


### 1. Сценарій багатокрокової діагностики (churn_predictor_v3)

Задача: виявити стан моделі відтоку клієнтів, обчислити ступінь порушення цільової затримки (SLO) та витягнути з регламенту інцидентів рекомендований інженерний протокол швидкого відновлення.

In [2]:
prompt = 'Diagnose churn_predictor_v3: check its status, calculate its latency SLO margin, and find the runbook mitigation protocol.'
print(f'User Prompt: {prompt}\n')

handler = agent.run(user_msg=prompt)
trace_log = []

async for event in handler.stream_events():
    if isinstance(event, ToolCall):
        print(f'  [TOOL CALL] {event.tool_name}({event.tool_kwargs})')
        trace_log.append({'event': 'ToolCall', 'tool': event.tool_name, 'kwargs': event.tool_kwargs})
    elif isinstance(event, ToolCallResult):
        print(f'  [TOOL RESULT] {event.tool_name} completed')
        trace_log.append({'event': 'ToolCallResult', 'tool': event.tool_name, 'output': str(event.tool_output)[:200]})

response = await handler
print(f'\nAgent Response:\n{response}')

User Prompt: Diagnose churn_predictor_v3: check its status, calculate its latency SLO margin, and find the runbook mitigation protocol.



  [TOOL CALL] list_degraded_models({})
  [TOOL RESULT] list_degraded_models completed


  [TOOL CALL] calculate_slo_margin({'metric_value': 315, 'slo_target': 250, 'metric_type': 'latency'})
  [TOOL RESULT] calculate_slo_margin completed


  [TOOL CALL] consult_incident_runbook({'query': 'mitigation protocol for degraded churn_predictor_v3 model with p99 latency 315ms and error rate 3.8%'})
  [TOOL RESULT] consult_incident_runbook completed



Agent Response:
The `churn_predictor_v3` model is currently in a degraded state with the following key metrics:

- **Latency SLO Violation**:  
  - P99 latency is **315ms**, exceeding the SLO target of **250ms**.  
  - The SLO margin is **26.0%**, indicating a clear violation.  
  - **Action**: Enable feature-cache fast-path and fall back to lightweight ensemble mode (depth=4) to reduce latency. Target recovery time: **5 minutes**.

- **Error Rate and Availability Degradation**:  
  - Error rate is **3.8%**, exceeding the 2% threshold.  
  - Availability is **98.62%**, below the 99.9% SLA.  
  - **Action**: Activate the imputed median fallback profile `churn_fallback_v3_imputed.bin` to handle missing batch features in user retention vectors.

### Mitigation Protocol (from Runbook):
1. **Immediate Actions**:
   - Apply the lightweight ensemble mode (depth=4) to reduce latency.
   - Activate `churn_fallback_v3_imputed.bin` to stabilize error rate.

2. **Escalation**:
   - If degradation

### 2. Сценарій критичної інфраструктурної аварії (llm_router_v1)

Задача: проаналізувати шлюз моделей `llm_router_v1`, зафіксувати сплеск помилок та затримки, визначити протокол екстреного відкату або перезапуску воркерів з chunked prefill.

In [3]:
prompt2 = 'Analyze llm_router_v1: check its error rate and availability, calculate its latency SLO margin, and retrieve the recovery protocol from the runbook.'
print(f'User Prompt: {prompt2}\n')

handler2 = agent.run(user_msg=prompt2)
trace_log2 = []

async for event in handler2.stream_events():
    if isinstance(event, ToolCall):
        print(f'  [TOOL CALL] {event.tool_name}({event.tool_kwargs})')
        trace_log2.append({'event': 'ToolCall', 'tool': event.tool_name, 'kwargs': event.tool_kwargs})
    elif isinstance(event, ToolCallResult):
        print(f'  [TOOL RESULT] {event.tool_name} completed')
        trace_log2.append({'event': 'ToolCallResult', 'tool': event.tool_name, 'output': str(event.tool_output)[:200]})

response2 = await handler2
print(f'\nAgent Response:\n{response2}')

User Prompt: Analyze llm_router_v1: check its error rate and availability, calculate its latency SLO margin, and retrieve the recovery protocol from the runbook.



  [TOOL CALL] list_degraded_models({})
  [TOOL RESULT] list_degraded_models completed


  [TOOL CALL] calculate_slo_margin({'metric_value': 1420, 'slo_target': 1000, 'metric_type': 'latency'})
  [TOOL RESULT] calculate_slo_margin completed


  [TOOL CALL] consult_incident_runbook({'query': 'recovery protocol for degraded llm_router_v1'})
  [TOOL RESULT] consult_incident_runbook completed



Agent Response:
### Analysis of `llm_router_v1`:

1. **Error Rate and Availability**:
   - Error Rate: **6.2%** (exceeds 2% threshold).
   - Availability: **97.4%** (below 99.9% SLA).
   - Both metrics indicate a degradation in service health, triggering the "Error Rate and Availability Degradation" protocol.

2. **Latency SLO Margin**:
   - P99 Latency: **1420 ms**.
   - SLO Target: **1000 ms**.
   - SLO Margin: **420 ms** (42.0% violation).
   - Status: **Violation**.
   - This indicates that the service is significantly exceeding its latency SLO.

3. **Recovery Protocol (from Runbook)**:
   - **Immediate Triage**:
     - Traffic surge check: Current requests per second (320 RPS) is below the threshold for vLLM models (>300 RPS), so no immediate scaling is triggered.
     - CPU/GPU saturation: Not explicitly reported, but the error rate suggests potential resource exhaustion.
   - **Mitigation Actions**:
     - Enable **speculative decoding**.
     - Enforce **strict max_tokens clam

### 3. Збережені структурні трейси виконання

Перевіримо наявність серіалізованих JSON-файлів у каталозі `outputs/traces/`.

In [4]:
trace_files = sorted(list(TRACES_DIR.glob('*.json')))
print(f'Знайдено збережених трейсів: {len(trace_files)}')
for tf in trace_files:
    with open(tf, 'r', encoding='utf-8') as f:
        tdata = json.load(f)
    print(f"- {tf.name}: tools={tdata.get('tools_called')}, elapsed={tdata.get('elapsed_sec')}s, correct={tdata.get('tool_selection_correct')}")

Знайдено збережених трейсів: 9
- scenario_01_direct_concept.json: tools=[], elapsed=7.05s, correct=True
- scenario_02_latency_metrics.json: tools=[], elapsed=11.27s, correct=True
- scenario_03_single_lookup.json: tools=['query_model_registry'], elapsed=10.32s, correct=True
- scenario_04_list_degraded.json: tools=['list_degraded_models'], elapsed=18.78s, correct=True
- scenario_05_multistep_churn.json: tools=['query_model_registry', 'calculate_slo_margin', 'consult_incident_runbook'], elapsed=38.14s, correct=True
- scenario_06_multistep_router.json: tools=['list_degraded_models', 'query_model_registry', 'calculate_slo_margin', 'consult_incident_runbook'], elapsed=50.74s, correct=True
- scenario_07_memory_continuation.json: tools=[], elapsed=22.79s, correct=False
- scenario_08_unknown_model.json: tools=['query_model_registry'], elapsed=5.22s, correct=True
- scenario_09_safety_refusal.json: tools=[], elapsed=2.95s, correct=True


### 4. Висновки до блокноту 2

- Агент продемонстрував стійку здатність до автономного багатоланкового планування (Multi-Step Tool Chaining), послідовно викликаючи реєстр, калькулятор та регламент інцидентів без жорстких умовних розгалужень у коді користувача.
- Реалізовано стрімінг подій за допомогою `stream_events()`, що уможливлює повний аудит проміжних кроків прийняття рішень.
- Усі трейси структуровано зафіксовані в JSON-форматі для подальшого оцінювання надійності.